# Sleep Stage Classification — CatBoost Experiment

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

# Preprocessing
eog_median = train['eog_burst_index'].median()
for df in [train, test]:
    df['eog_burst_missing'] = df['eog_burst_index'].isnull().astype(int)
    df['eog_burst_index']   = df['eog_burst_index'].fillna(eog_median)

FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
X = train[FEATURES].values
y = train['sleep_stage'].values

print(f'train: {train.shape} | test: {test.shape}')
print(f'Признаков: {len(FEATURES)}')
print(f'Пропуски eog_burst_index — train: {train["eog_burst_index"].isnull().sum()} | test: {test["eog_burst_index"].isnull().sum()}')

train: (9000, 24) | test: (5000, 23)
Признаков: 22
Пропуски eog_burst_index — train: 0 | test: 0


In [2]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

model_cb = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=50
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cb_scores = cross_val_score(model_cb, X, y, cv=cv, scoring='f1_macro')

print('=' * 50)
print('CatBoost — cross-val результаты')
print('=' * 50)
print(f'  Folds : {cb_scores.round(4)}')
print(f'  Mean  : {cb_scores.mean():.4f}')
print(f'  Std   : {cb_scores.std():.4f}')

ModuleNotFoundError: No module named 'catboost'

In [3]:
# Обучение на полном train и генерация предсказаний
model_cb_final = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=50
)
model_cb_final.fit(X, y)

preds = model_cb_final.predict(test[FEATURES].values).astype(int)
submission = pd.DataFrame({'id': test['id'], 'sleep_stage': preds})
submission.to_csv('submission_catboost.csv', index=False)

print('=== Первые 5 строк ===')
print(submission.head().to_string(index=False))
print('\n=== value_counts предсказаний ===')
print(submission['sleep_stage'].value_counts().sort_index())
print(f'\nСохранено: submission_catboost.csv  ({len(submission)} строк)')

NameError: name 'CatBoostClassifier' is not defined